# NB_{{DOM}}_Gold · Phase 2 · Silver → {{LH}} star schema
Column names here must match `sec.ColumnRule` exactly (DAT-04). Every fact keeps `org_id` for RLS (DAT-03).

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "{{LH}}"}}

In [ ]:
DOMAIN = "{{DOM}}"
ONELAKE = "abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}/Tables/{path}"
SRC = {
  "own_silver": ("{{WS_OWN_SILVER}}", "{{LH_OWN_SILVER}}"),
  "ap_silver": ("{{WS_AP_SILVER}}", "{{LH_AP_SILVER}}"),
  "own_bronze": ("{{WS_OWN_BRONZE}}", "{{LH_OWN_BRONZE}}"),
}
ORG_COUNTRY = {101: "Kuwait", 102: "Saudi Arabia", 103: "United Arab Emirates"}
ORG_CCY = {101: "KWD", 102: "SAR", 103: "AED"}

In [ ]:
from pyspark.sql import functions as F
def view(src, path, name):
    ws, lh = SRC[src]
    spark.read.format("delta").load(ONELAKE.format(ws=ws, lh=lh, path=path)).createOrReplaceTempView(name)
view("ap_silver", "silver/supplier", "s_supplier")
view("own_bronze", "ebs/hr_operating_units", "b_hr")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"gold.{name}")
    print(f"gold.{name}: {spark.table('gold.' + name).count()} rows")

In [ ]:
# Conformed dimensions (identical keys in both domains)
spark.sql("SELECT * FROM VALUES " + ", ".join(f"({k}, '{ORG_COUNTRY[k]}', '{ORG_CCY[k]}')" for k in ORG_COUNTRY) + " AS t(org_id, country, currency_code)").createOrReplaceTempView("org_ref")
save(spark.sql("""SELECT CAST(h.ORGANIZATION_ID AS INT) org_id, h.NAME org_name, r.country, r.currency_code
                  FROM b_hr h JOIN org_ref r ON r.org_id = CAST(h.ORGANIZATION_ID AS INT)"""), "dim_org")
save(spark.sql("""
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) date_key, d AS date, YEAR(d) year, QUARTER(d) quarter, MONTH(d) month,
       date_format(d, 'MMMM') month_name, date_format(d, 'EEEE') day_of_week,
       CASE WHEN date_format(d, 'E') IN ('Fri', 'Sat') THEN true ELSE false END is_weekend
FROM (SELECT explode(sequence(DATE'2025-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) d)"""), "dim_date")
save(spark.sql("""
SELECT CAST(ROW_NUMBER() OVER (ORDER BY supplier_id, valid_from) AS INT) supplier_key, supplier_id, supplier_name, category, country,
       vat_registration_num, payment_terms, is_current, valid_from, valid_to
FROM s_supplier"""), "dim_supplier")
spark.table("gold.dim_supplier").filter("is_current").createOrReplaceTempView("cur_supplier")

In [ ]:
if DOMAIN == "PRC":
    for t in ["po_header", "po_line", "receipt", "item"]:
        view("own_silver", f"silver/{t}", f"s_{t}")
    save(spark.sql("""SELECT CAST(ROW_NUMBER() OVER (ORDER BY item_id) AS INT) item_key, item_id, item_code, item_name, uom,
                             category_name AS category FROM s_item"""), "dim_item")
    save(spark.sql("""
    SELECT l.po_line_id AS po_line_key, h.po_number, l.line_number, l.org_id, s.supplier_key, i.item_key,
           CAST(date_format(h.order_date, 'yyyyMMdd') AS INT) order_date_key, CAST(date_format(l.need_by_date, 'yyyyMMdd') AS INT) need_by_date_key,
           l.quantity, l.uom, l.unit_price, l.line_amount AS po_value, l.currency_code, l.line_amount_kwd AS po_value_kwd,
           CASE WHEN h.closed_status = 'Closed' THEN 'CLOSED' WHEN h.approval_status = 'Approved' THEN 'OPEN' ELSE 'DRAFT' END AS status
    FROM s_po_line l JOIN s_po_header h ON h.po_header_id = l.po_header_id
    LEFT JOIN cur_supplier s ON s.supplier_id = h.supplier_id JOIN gold.dim_item i ON i.item_id = l.item_id"""), "fact_purchase_order")
    save(spark.sql("""
    SELECT r.transaction_id AS receipt_key, r.receipt_number, r.org_id, r.po_line_id AS po_line_key, f.supplier_key, f.item_key,
           CAST(date_format(r.transaction_date, 'yyyyMMdd') AS INT) receipt_date_key, r.transaction_type, r.quantity
    FROM s_receipt r JOIN gold.fact_purchase_order f ON f.po_line_key = r.po_line_id"""), "fact_receipt")

In [ ]:
if DOMAIN == "AP":
    for t in ["invoice", "payment", "bank_account"]:
        view("own_silver", f"silver/{t}", f"s_{t}")
    save(spark.sql("""SELECT b.bank_account_id AS bank_account_key, s.supplier_key, b.bank_name, b.bank_account_num, b.iban, b.currency_code
                      FROM s_bank_account b LEFT JOIN cur_supplier s ON s.supplier_id = b.supplier_id"""), "dim_bank_account")
    save(spark.sql("""
    SELECT i.invoice_id AS invoice_key, i.invoice_number, i.org_id, s.supplier_key,
           CAST(date_format(i.invoice_date, 'yyyyMMdd') AS INT) invoice_date_key, CAST(date_format(i.due_date, 'yyyyMMdd') AS INT) due_date_key,
           i.invoice_amount, i.currency_code, i.invoice_amount_kwd, i.amount_remaining, i.payment_status, i.approval_status
    FROM s_invoice i LEFT JOIN cur_supplier s ON s.supplier_id = i.supplier_id"""), "fact_invoice")
    save(spark.sql("""
    SELECT p.invoice_payment_id AS payment_key, p.check_number, p.org_id, s.supplier_key, p.invoice_id AS invoice_key,
           p.bank_account_id AS bank_account_key, CAST(date_format(p.payment_date, 'yyyyMMdd') AS INT) payment_date_key,
           p.amount, p.currency_code, p.amount_kwd
    FROM s_payment p LEFT JOIN cur_supplier s ON s.supplier_id = p.supplier_id"""), "fact_payment")

In [ ]:
# Reconciliation check: Gold totals equal Silver totals (P2 exit criterion)
if DOMAIN == "PRC":
    a = spark.sql("SELECT ROUND(SUM(po_value_kwd),3) v FROM gold.fact_purchase_order").first().v
    b = spark.sql("SELECT ROUND(SUM(line_amount_kwd),3) v FROM s_po_line").first().v
else:
    a = spark.sql("SELECT ROUND(SUM(invoice_amount_kwd),3) v FROM gold.fact_invoice").first().v
    b = spark.sql("SELECT ROUND(SUM(invoice_amount_kwd),3) v FROM s_invoice").first().v
assert a == b, f"Gold {a} != Silver {b}"
print("Reconciled, total KWD:", a)